<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Profile before changing

**[Profile before changing](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/)** · SQL and Data Preparation for AI · Module 4, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure the problems in the help-desk data before you change any value: missing values and what they mean, sentinel values, duplicates, invalid categories, mixed units and values outside a sensible range. You profile in SQL and in pandas, and you write the results in a profile.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account and no local install: the first cell installs pandas 3.0.6 in this notebook. |
| **You should know** | `COUNT`, `GROUP BY` and `IS NULL`, from [Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/). Left joins, from [Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/). What the extract is, from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/). |
| **You do not need** | The local project from Modules 1 to 3. The setup below builds the same database and the same extract here. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M04-L01-profile-before-changing/profile-before-changing-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `extract.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. What a profile is

Larkfield's help desk wants a model that predicts if a ticket will need a senior agent. In this module, Mei, the data engineer, cleans the extract from Module 3. Before she changes a value, she **profiles** the data.

A **profile** is a short description of a dataset, column by column: how many values are missing, which values occur, and the smallest and largest value. It only finds and counts. It changes nothing. It is the "before" picture that you compare with after cleaning.

The next cell makes five made-up tickets in memory, `tiny`. Every row has a problem that causes no error:

- T-2 spells the channel `Email`, and has no `order_value` (NULL).
- T-3 has `order_value` `-1`, a code for "unknown". T-3 is in the table twice.
- T-4 has 4250 cents, not dollars, 31,000 words, and the year 2036.

Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE tickets (ticket_id TEXT, channel TEXT, order_value REAL, word_count INTEGER, created_at TEXT);
INSERT INTO tickets VALUES
    ('T-1', 'email', 42.50, 80, '2026-03-02'),
    ('T-2', 'Email', NULL, 45, '2026-03-02'),
    ('T-3', 'chat', -1, 120, '2026-03-03'),
    ('T-3', 'chat', -1, 120, '2026-03-03'),
    ('T-4', 'web_form', 4250, 31000, '2036-03-04');
""");

> **Predict.** What does `AVG(order_value)` return on this table? Is it close to a normal order of about 42 dollars? Write it down, then run the cell.

In [ ]:
sql("SELECT COUNT(*) AS ticket_rows, AVG(order_value) AS avg_order_value FROM tickets;", db=tiny)

> **Check.** You should see 5 rows and an average of **1072.625**. `AVG` skips the NULL and uses the other four values: (42.5 − 1 − 1 + 4250) / 4. The query runs without an error, and the number is wrong.

A profile asks five questions. Each question finds one kind of problem:

| Question | Kind of problem | In the five tickets |
|---|---|---|
| Which values are missing, and what does each one mean? | **missingness** | T-2 has no `order_value` |
| Does a row, or a key, occur more than once? | **duplicates** | T-3 twice |
| Are all values in the list of allowed values? | **invalid categories** | `Email` next to `email` |
| Do all values use the same unit? | **inconsistent units** | 4250 cents next to 42.50 dollars |
| Are the values in a sensible range? | **suspicious ranges** | 31000 words, the year 2036, `-1` |

## 2. Missingness

Mei profiles the full export in `larkfield.db` (1,417 ticket rows), not only the extract. `COUNT(*)` counts rows. `COUNT(column)` counts the values that are not NULL. So `COUNT(*) - COUNT(column)` is the number of missing values.

> **Predict.** Which column has the most NULLs? Then run the cell.

In [ ]:
sql("""
SELECT
    COUNT(*)                      AS ticket_rows,
    COUNT(*) - COUNT(customer_id) AS no_customer_id,
    COUNT(*) - COUNT(channel)     AS no_channel,
    COUNT(*) - COUNT(order_value) AS no_order_value,
    COUNT(*) - COUNT(word_count)  AS no_word_count,
    COUNT(*) - COUNT(closed_at)   AS no_closed_at
FROM tickets;
""")

> **Check.** You should see 1 row: `1417  12  38  188  0  12`.

A count is not enough. The profile must also say what each NULL **means**. Grace, the support lead, explains three of them: a missing `customer_id` is a guest checkout, a missing `channel` was not recorded, and a missing `closed_at` is a ticket that is still open. What does a missing `order_value` mean?

> **Your turn.** Write a query that groups the tickets with a NULL `order_value` by `team`: the team and the number of tickets. Use `IS NULL`, not `= NULL`. Then run the check cell.

In [ ]:
no_order = """
-- Which teams have tickets with no order_value?
-- Show each team and its number of such tickets.
-- Write your query here
"""
sql(no_order)

In [ ]:
check(no_order, rows=1, columns=2, expected="412383c6fc65dc81")

When the check passes, you see one row: `account`, 188. Every account ticket has a NULL `order_value`, and no other ticket has one. Grace explains: "Account questions, such as a password reset, have no order." Here NULL does **not** mean "unknown". It means "there is no order".

## 3. Missing values that are not NULL

A system can also mark a missing value with a normal-looking value, such as `-1`, `0`, `9999` or `N/A`. This is a **sentinel value**. A NULL count does not find it. The most common values often do. Run the cell.

In [ ]:
sql("""
SELECT order_value, COUNT(*) AS n
FROM tickets
GROUP BY order_value
ORDER BY n DESC
LIMIT 5;
""")

> **Check.** You should see `NULL` with 188 rows first, then `-1.0` with **11** rows. Real amounts occur at most 3 times each.

An order cannot cost minus one dollar. Grace confirms: the order system writes `-1` when it cannot find the order. So `order_value` has two kinds of missing value: 188 NULLs that mean "no order", and 11 sentinels that mean "unknown order".

## 4. Duplicates: exact rows and repeated keys

An **exact duplicate** is a row that is equal to another row in every column. A **repeated key** is a key, here `ticket_id`, that occurs in more than one row, even if other columns differ.

> **Predict.** The table has 1,417 rows and 1,403 distinct ticket IDs. How many distinct rows does `SELECT DISTINCT *` give? Then run the cell.

In [ ]:
sql("""
SELECT
    COUNT(*)                  AS ticket_rows,
    COUNT(DISTINCT ticket_id) AS distinct_ids,
    (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM tickets)) AS distinct_rows
FROM tickets;
""")

> **Check.** You should see `1417  1403  1403`. The two counts are equal, so all 14 extra rows are exact copies (an export page was sent twice). Removing a copy loses no information. If the counts were different, some tickets would have two **different** rows, and you would need to decide which row is correct.

## 5. Invalid categories

A category column must use a fixed list of values. For `channel`, the list is `email`, `chat`, `phone` and `web_form`.

> **Predict.** How many different values does `channel` have? Then run the cell.

In [ ]:
sql("""
SELECT channel, COUNT(*) AS n
FROM tickets
GROUP BY channel
ORDER BY n DESC;
""")

> **Check.** You should see 9 rows. The list of four plus NULL accounts for 1,377 rows. The other 40 rows use four other spellings: `Email` (20), ` chat` with a space at the start (8), `EMAIL` (7) and `webform` (5). The space in ` chat` is hard to see, but `GROUP BY` shows it as a separate row.

> **Your turn.** The allowed segments are `home`, `trade` and `business`. Write a query that lists each spelling of `segment` in the `customers` table, with the number of customers that use it. Then run the check cell. Which spelling is not allowed?

In [ ]:
segments = """
-- List each spelling of segment in the customers table,
-- with the number of customers that use it.
-- Write your query here
"""
sql(segments)

In [ ]:
check(segments, rows=4, columns=2, expected="6ae654e4f52e8c87")

When the check passes, you see 4 rows: `home` 166, `trade` 47, `business` 24 and `Trade` 3. `Trade` is the same segment as `trade`, with a capital letter.

## 6. Inconsistent units

A column can have one name and two units. To find this, compare groups: if one group has values about 100 times larger than the others, suspect a different unit. Mei groups the `web_form` tickets by month.

> **Predict.** Does the average order stay about the same every month? Then run the cell.

In [ ]:
sql("""
SELECT
    strftime('%Y-%m', created_at)  AS month,
    COUNT(*)                       AS n,
    ROUND(AVG(order_value), 2)     AS avg_value,
    MAX(order_value)               AS max_value
FROM tickets
WHERE channel = 'web_form'
  AND order_value > 0
GROUP BY month
ORDER BY month;
""")

> **Check.** In January, the average web form order is **7957.91** dollars. From February to June, it is between 88.76 and 158.07. The last row, `2036-01`, is one ticket with a wrong year.

> **Your turn: change one thing.** Copy the query into the next cell, and change `'web_form'` to `'email'`. Run it. Do you see the same jump in January?

In [ ]:
sql("""
-- The same query for the email channel. Write it here.
""")

> **Check.** Email orders stay between about 100 and 145 dollars in every month. Only the web form jumps. Grace knows the reason: until 1 February 2026, the old web form sent the amount in **cents**. A model that sees these values learns that web form customers buy expensive things, which is false.

## 7. Suspicious ranges

The smallest and largest value of each column show values that are impossible or very unlikely. Run the cell. For each row, ask: "Is this value possible?"

In [ ]:
sql("""
SELECT 'created_at' AS column_name, MIN(created_at) AS smallest, MAX(created_at) AS largest FROM tickets
UNION ALL
SELECT 'priority', MIN(priority), MAX(priority) FROM tickets
UNION ALL
SELECT 'order_value', MIN(order_value), MAX(order_value) FROM tickets
UNION ALL
SELECT 'word_count', MIN(word_count), MAX(word_count) FROM tickets;
""")

> **Check.** You should see 4 rows:

- `created_at` ends at **2036-01-24 21:27:43**: someone typed 2036 for 2026. The extract leaves this row out.
- `priority` is 1 to 3, as expected.
- `order_value` runs from **-1** (the sentinel) to **34518** (cents).
- `word_count` runs from 0 to **48210**. Grace: 0 words is a ticket with only an attachment (valid). 48,210 words is `T-10361`, where the customer pasted a log file.

Some problems are visible only when you compare two tables. A ticket cannot exist before the customer's account.

> **Your turn.** Add a `WHERE` clause that keeps a ticket only if its day, `date(t.created_at)`, is before `c.joined_on`. Predict the number of rows first. Then run the check cell.

In [ ]:
before_joined = """
-- Find the tickets that were created before the customer joined.
-- Tip: date(created_at) gives the day part of the timestamp.
SELECT t.ticket_id, t.customer_id, t.created_at, c.joined_on
FROM tickets AS t
JOIN customers AS c
    ON c.customer_id = t.customer_id
-- add a WHERE clause here
;
"""
sql(before_joined)

In [ ]:
check(before_joined, rows=3, columns=4, expected="50bf9c7a0d4ff5ab")

When the check passes, you see 3 rows: `T-11185`, `T-11224` and `T-11375`. Either the ticket time or the join date is wrong, and the data does not say which one. Mei cannot correct these rows with confidence.

On your computer, Mei saves each profile query in `queries/`, for example `queries/41-profile-missing.sql`, and runs it with `python3 q.py queries/41-profile-missing.sql`. The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/#run-the-profile-on-your-computer) shows how.

## 8. The same profile in pandas

The rest of this module uses **pandas**, a Python library for tables. A pandas table is a **DataFrame**: rows and named columns, like a SQL table, in the memory of your Python program. The setup installed pandas 3.0.6 here. On your computer, you install it in a virtual environment, as the [lesson page](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/#the-same-profile-in-pandas) shows.

Mei's pandas profile reads the extract, `data/extract/tickets.csv`, the file that the cleaning step uses. The next cell saves her script as `scripts/profile.py`, as on your computer. Run it. It prints `Writing scripts/profile.py`.

In [ ]:
%%writefile scripts/profile.py
"""Profile the extracted tickets. Print counts; change nothing."""

import pandas as pd

df = pd.read_csv("data/extract/tickets.csv")
print(f"Rows: {len(df)}")

missing = df.isna().sum()
print("\nMissing values (columns with at least one):")
print(missing[missing > 0])

print(f"\nExact duplicate rows: {df.duplicated().sum()}")
print(f"Rows with a ticket_id seen before: {df.duplicated(subset='ticket_id').sum()}")

print("\nChannel values:")
print(df["channel"].value_counts(dropna=False))

print("\nRanges:")
print(df[["order_value", "word_count"]].describe().round(2))
print(f"\norder_value = -1: {(df['order_value'] == -1).sum()}")
print(f"order_value > 2000: {(df['order_value'] > 2000).sum()}")
print(f"word_count > 10000: {(df['word_count'] > 10000).sum()}")

| Code | What it does | Same as in SQL |
|---|---|---|
| `pd.read_csv(...)` | Reads the CSV file into a DataFrame. | the table |
| `df.isna().sum()` | Counts the missing values in each column. | `COUNT(*) - COUNT(col)` |
| `df.duplicated()` | Marks each row that is an exact copy of an earlier row. | `COUNT(*)` minus `SELECT DISTINCT *` |
| `df.duplicated(subset='ticket_id')` | The same, but compares only `ticket_id`. | `COUNT(*) - COUNT(DISTINCT ticket_id)` |
| `value_counts(dropna=False)` | Counts each value, and includes the missing values. | `GROUP BY channel` |
| `describe()` | Count, mean, standard deviation, minimum, quartiles and maximum. | `MIN`, `MAX`, `AVG` |

> **Predict.** The database has 1,417 ticket rows. How many rows does the extract have? Then run the script.

In [ ]:
!"{sys.executable}" scripts/profile.py

> **Check.** You should see `Rows: 1416`, `Exact duplicate rows: 14`, a channel list with `NaN 38`, a mean `order_value` of 272.09 and a median (`50%`) of 88.79, then `order_value = -1: 11`, `order_value > 2000: 24` and `word_count > 10000: 1`.

Compare it with the SQL profile. **Count everything** includes the differences:

- 1,416 rows, not 1,417: the extract left out the 2036 ticket. That was a `web_form` ticket with 34,518 cents, so `web_form` has 199 rows here, not 200, and `order_value > 2000` gives 24, not 25.
- pandas shows a missing value as `NaN` ("not a number").
- `segment`, `region` and `joined_on` come from the customer join in the extract. They are missing for 21 rows: the 12 guest tickets and the 9 tickets whose customer is not in the customer table. `region` has 118 more missing values, because 9 customers have no region.
- The mean `order_value` is 272.09 dollars, but the median is 88.79. The cents and the `-1` values pull the mean. Do not report this mean.

## 9. Guided practice: write the profile

The profile is a document, not only output. Priya, the data scientist, reads it before she trains a model.

First make the folder `reports` in the project. Run the cell. It prints nothing.

In [ ]:
Path("reports").mkdir(exist_ok=True)

> **Your turn.** Complete `reports/profile.md` in the next cell, then run it.
>
> 1. Write the row count of the source at the top.
> 2. In each section, write the count and the query or code that gives it.
> 3. For each missing value and each strange value, write what it means. If you do not know, write the question for Grace.
> 4. Under "Not decided yet", list each problem without a fix.
>
> Then run the check cell. It looks for a count or an ID for each problem, not for your exact words.

In [ ]:
%%writefile reports/profile.md
# Profile: Larkfield tickets

Source: data/extract/tickets.csv, ... rows. Snapshot 2026-07-01 06:00.
Nothing changed.

## Missing values
(each column, its count, and what a missing value means)

## Duplicates

## Categories

## Units

## Ranges

## Not decided yet

In [ ]:
# Check: does the profile give a count for each problem? (It does not check your words.)
import re
text = Path("reports/profile.md").read_text(encoding="utf-8").replace(",", "")
needed = {
    "the rows in the extract": "1416",
    "the account tickets with no order_value": "188",
    "the -1 values": "11",
    "the exact duplicates": "14",
    "the other channel spellings": "40",
    "the order values in cents": "24",
    "the ticket with a pasted log file": "T-10361",
    "the tickets created before the customer joined": "T-11185",
}
absent = [what for what, value in needed.items() if not re.search(rf"(?<![\w-]){value}(?![\w-])", text)]
print("Check passed." if not absent else "Not yet: the profile has no count or ID for " + "; ".join(absent) + ".")

## 10. Your turn: profile the outcome events

Profile the `outcomes` table with the same five questions.

> **Your turn.** Write a query that returns one row with three columns: the number of event rows, the number of missing `resolution_code` values, and the number of missing `csat` values. Then run the check cell.

In [ ]:
outcome_missing = """
-- Missing values in outcomes: event rows, no resolution_code, no csat.
-- Write your query here
"""
sql(outcome_missing)

In [ ]:
check(outcome_missing, rows=1, columns=3, expected="2db0b7bd8227e757")

When the check passes, you see `1874  341  959`. `resolution_code` is set only for `resolved` events, and only `resolved` events have a `csat` score. So these NULLs mean "not resolved in this event" and "the customer did not answer the survey".

> **Predict.** Does `outcomes` have exact duplicate rows? Run the cell to find out.

In [ ]:
sql("""
SELECT
    COUNT(*)                   AS event_rows,
    COUNT(DISTINCT outcome_id) AS distinct_ids,
    (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM outcomes)) AS distinct_rows
FROM outcomes;
""")

> **Check.** You should see `1874  1874  1874`: no exact duplicates, and no repeated `outcome_id`. But an event is "this ticket, at this time, got this status". The ID does not say what the event is.

> **Your turn.** Write a query that returns one number, `repeated_events`: the number of event rows minus the number of distinct (`ticket_id`, `recorded_at`, `status`) combinations. Then run the check cell.

In [ ]:
repeated_events = """
-- Events logged twice: same ticket, time and status, new outcome_id.
-- Write your query here
"""
sql(repeated_events)

In [ ]:
check(repeated_events, rows=1, columns=1, expected="90c7a32d0fead02b")

When the check passes, the number is **30**: 30 events were logged twice with a new `outcome_id`. A duplicate check must use the columns that say what the event is, not only the ID.

The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/#your-turn-profile-the-outcome-events) has the expected profile of `outcomes`, with the orphan events and the ranges. Compare your results with it.

## 11. Failure case: a NULL count misses the sentinels

Tomás counts the missing order values with `isna()`. Run the cell.

In [ ]:
import pandas as pd
extract = pd.read_csv("data/extract/tickets.csv")
order_value = extract["order_value"]
print("missing (isna):", order_value.isna().sum())
print("sentinel -1:", (order_value == -1).sum())
print("mean with the -1 values:", round(order_value[order_value < 2000].mean(), 2))
print("mean without them:", round(order_value[(order_value < 2000) & (order_value != -1)].mean(), 2))

> **Check.** You should see 188 missing values and 11 sentinels: 199 orders without a known amount, not 188. Both means leave out the cents values (above 2,000). With the `-1` values, the mean is 122.79; without them, 123.93. The difference is small here, and nothing fails.

**Cause:** for SQL and pandas, `-1` is a normal number. A NULL count does not see it, and `AVG` and `mean()` use it as minus one dollar.

**Fix:** look at the smallest values and the most common values of every number column, not only the NULL count. Look for typical sentinels (`-1`, `0`, `999`, `9999`, `N/A`, `none`, `unknown`). Ask the owner of the data what each one means, and write the answer in the profile.

**In AI work:** a model trained on `-1` learns that some orders cost minus one dollar. If "unknown order" is related to escalation, the model links escalation to a very small order, which is not true.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/). They count toward your certificate when you are signed in and enrolled.

In this lesson you profiled the data before you changed it. You counted the missing values and wrote what each one means, found the `-1` sentinels, told exact duplicates from repeated keys, found the other spellings, the cents and the impossible values, and wrote everything in `reports/profile.md`.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Transform with Python](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.